In [13]:
import os
import win32com.client
import pandas as pd
from datetime import datetime

outlook_app = win32com.client.Dispatch('Outlook.Application')
PATH_ANEXOS = fr'{os.getcwd()}\anexos'

def substituir_variaveis(var, texto):
    return(eval("f'{}'".format(texto)))

def selecionar_conta_email(email):
    for account in outlook_app.Session.Accounts:
        if account.DisplayName == email:
            return(account)
        
def salvar_planilhas(df_dest, df_msg, df_anexos):
    with pd.ExcelWriter(fonte, datetime_format='DD/MM/YYYY') as writer:  
        df_dest.to_excel(writer, sheet_name='Destinatarios', index=False)
        df_msg.to_excel(writer, sheet_name='Mensagem', index=False)
        df_anexos.to_excel(writer, sheet_name='Anexos', index=False)
        
def escrever_mensagem(destinatario, df_msg, df_anexos):
    remetente = df_msg['Remetente'][0]    
    remetente = selecionar_conta_email(remetente)
    if remetente is None:
        print('\t[ERRO] Conta de remetente no Outlook não encontrada')
        raise Exception('Conta de remetente no Outlook não encontrada')
    
    
    msg = outlook_app.CreateItem(0)   
    msg._oleobj_.Invoke(*(64209, 0, 8, 0, remetente))
    msg.ReadReceiptRequested = True

    msg.To = destinatario['PontoFocalEmail']

    assunto = df_msg['Assunto'][0]
    assunto = substituir_variaveis(destinatario, assunto)
    msg.Subject = assunto

    msg.BodyFormat = 2
    txt = ''
    for paragrafo in df_msg['Paragrafos']:
        txt += '<p>' + paragrafo.strip() + '</p>'

    txt = substituir_variaveis(destinatario, txt)
    msg.HTMLBody = txt

    for idx, row in df_anexos.iterrows():
        anexo = row['Anexos']
        anexo = substituir_variaveis(destinatario, anexo)
        anexo = f'{PATH_ANEXOS}\\{anexo}'                
        obrigatorio = row['Obrigatório']
        if (obrigatorio.upper() == 'N') and (not os.path.isfile(anexo)):
            continue
            
        print(f'Anexando {anexo}')
        msg.Attachments.Add(anexo)

    return(msg)

def enviar_emails(fonte):
    print(f"Processando planilha: {fonte}")
    
    df_dest = pd.read_excel(fonte, 'Destinatarios')
    df_msg  = pd.read_excel(fonte, 'Mensagem')
    df_anexos = pd.read_excel(fonte, 'Anexos')
    
    df_mail = df_dest[~df_dest['PontoFocalEmail'].isna()]
    df_mail = df_mail[df_mail['Envio'].isna()]
    for col in df_mail.select_dtypes(include=["datetime64[ns]", "datetime64"]).columns:
        df_mail[col] = df_mail[col].dt.strftime("%d/%m/%Y")
        
    for i, row in df_mail.iterrows():
        try:
            mensagem = escrever_mensagem(row, df_msg, df_anexos)            
            
            #mensagem.Display()
            #break
            
            mensagem.Send()
            df_dest.loc[df_dest.index == i, 'Envio'] = datetime.today()
            print(f"\tEnviado e-mail para {row['Orgao']}")
            salvar_planilhas(df_dest, df_msg, df_anexos)
        except Exception as e:
            print(f"\tErro ao enviar e-mail para {row['Orgao']}", type(e), e)
            print(e)

path_fonte = './fonte/'
for fonte in os.listdir(path_fonte):
    fonte = path_fonte + fonte
    if fonte.endswith('.xlsx'):
        enviar_emails(fonte)

Processando planilha: ./fonte/01-TSID01 Reiteracao.xlsx
Processando planilha: ./fonte/~$01-TSID03.xlsx


ValueError: Excel file format cannot be determined, you must specify an engine manually.